# 📘 The AI Engineer's LLM Workbook

**14 Chapters · 14 Google Colab Notebooks · Beginner to Production**

---

*© 2026 JAWNVION LLC — www.jawnvion.com — peter@jawnvion.com*

*Licensed for individual use. Do not redistribute.*

---

## What's Inside

| # | Chapter |
|---|---------|
| 01 | AI Fundamentals & Problem Framing |
| 02 | Data Science Toolkit (NumPy, Pandas, Matplotlib) |
| 03 | Neural Networks from Scratch |
| 04 | Transformers Architecture Deep Dive |
| 05 | HuggingFace & Pre-Trained Models |
| 06 | QLoRA Fine-Tuning |
| 07 | DPO Alignment Training |
| 08 | Retrieval-Augmented Generation (RAG) |
| 09 | Model Evaluation & Benchmarking |
| 10 | FastAPI Deployment |
| 11 | Monitoring & Observability |
| 12 | Security for AI Systems |
| 13 | Cost Optimization & Quantization |
| 14 | Capstone: End-to-End LLM Project |

---

> **How to use:** Click **Runtime → Run All** in Google Colab, or run cells one at a time.
> Each chapter builds on the last — complete them in order for best results.

---


# Chapter 1: AI & Machine Learning Fundamentals
**JAWNVION LLC — AI Training Workbook**

Before you can fine-tune a language model, you need to understand what a model *is* —
and what it means to *train* one. This chapter builds that foundation from first
principles using nothing but Python and numpy.

**What you'll learn:**
- The difference between AI, Machine Learning, and Deep Learning
- How a machine "learns" — the training loop explained step by step
- What a loss function is and why it matters
- Gradient descent: how models improve themselves automatically
- Build and train a linear regression model from scratch (no libraries)
- Understand overfitting, underfitting, and the train/validation split

**No GPU required.** Everything in this chapter runs on CPU in under 2 minutes.

In [ ]:
# — Cell 1: Environment Check ——————————————————————————
import sys
import numpy as np

print(f"✓  Python  : {sys.version.split()[0]}")
print(f"✓  NumPy   : {np.__version__}")
print()
print("No GPU required for this chapter — all computations run on CPU.")
print("These fundamentals apply equally whether you later train on T4 or A100.")

In [ ]:
# — Cell 2: The AI / ML / DL Landscape ————————————————
# Three terms you'll hear constantly — here's what they actually mean.

landscape = {
    "Artificial Intelligence (AI)": {
        "definition": "Any technique that enables machines to mimic human intelligence.",
        "examples":   ["Rule-based chess engines", "Expert systems", "Neural networks"],
        "scope":      "Broadest — includes everything below",
    },
    "Machine Learning (ML)": {
        "definition": "AI systems that LEARN patterns from data instead of following hard-coded rules.",
        "examples":   ["Spam filters", "Recommendation engines", "GPT-style LLMs"],
        "scope":      "Subset of AI — requires data + optimization",
    },
    "Deep Learning (DL)": {
        "definition": "ML using multi-layer neural networks capable of learning hierarchical representations.",
        "examples":   ["Image classifiers", "TinyLlama", "Stable Diffusion"],
        "scope":      "Subset of ML — requires large data + GPU compute",
    },
}

for name, info in landscape.items():
    print(f"{'='*60}")
    print(f"  {name}")
    print(f"  Definition : {info['definition']}")
    print(f"  Examples   : {', '.join(info['examples'])}")
    print(f"  Scope      : {info['scope']}")
print('='*60)
print()
print("Key insight: Every LLM (including TinyLlama) is Deep Learning,")
print("which is Machine Learning, which is Artificial Intelligence.")

In [ ]:
# — Cell 3: The Training Loop — The Heart of ML ————————
# Every ML model — from linear regression to GPT-4 — trains the same way.
# Understanding this loop makes every other concept in this course click.

print("THE MACHINE LEARNING TRAINING LOOP")
print("=" * 50)
print()
steps = [
    ("1. DATA",        "Collect labelled examples  (input → correct output)"),
    ("2. PREDICT",     "Pass input through the model → get a prediction"),
    ("3. MEASURE",     "Compare prediction to correct answer → compute LOSS"),
    ("4. LEARN",       "Adjust model weights to reduce loss → GRADIENT DESCENT"),
    ("5. REPEAT",      "Go back to step 2 with the next batch of data"),
    ("6. EVALUATE",    "After N epochs, check accuracy on UNSEEN data"),
]
for step, desc in steps:
    print(f"  {step:<12}  {desc}")

print()
print("This loop is called an EPOCH when it covers the full dataset once.")
print("LLM pre-training runs this loop for BILLIONS of iterations.")
print()
print("The thing being adjusted in step 4 is called the model's WEIGHTS")
print("(also called parameters). TinyLlama has 1.1 BILLION of them.")

In [ ]:
# — Cell 4: Loss Functions — Measuring 'How Wrong' ————
import numpy as np

# Suppose we're predicting house prices.
# y_true  = actual prices (in $100k)
# y_pred  = our model's guesses

y_true = np.array([3.0, 5.0, 2.5, 7.0, 4.5])
y_pred = np.array([2.8, 5.5, 2.0, 8.0, 4.0])

# ── Mean Squared Error (MSE) ─────────────────────────────
# The most common loss for regression.  Penalises big errors heavily.
mse = np.mean((y_true - y_pred) ** 2)

# ── Mean Absolute Error (MAE) ────────────────────────────
# Less sensitive to outliers than MSE.
mae = np.mean(np.abs(y_true - y_pred))

# ── Root Mean Squared Error (RMSE) ───────────────────────
# Same units as the target — easier to interpret.
rmse = np.sqrt(mse)

print("Loss Function Demo — House Price Prediction")
print("=" * 45)
print(f"  Actual prices    : {y_true}")
print(f"  Predicted prices : {y_pred}")
print(f"  Errors           : {np.round(y_true - y_pred, 2)}")
print()
print(f"  MSE  (Mean Squared Error)       : {mse:.4f}")
print(f"  MAE  (Mean Absolute Error)      : {mae:.4f}")
print(f"  RMSE (Root Mean Squared Error)  : {rmse:.4f}")
print()
print("Key insight: training = minimising the loss.")
print("When loss → 0, predictions → perfect.")
print()
print("For LLMs, the loss is CROSS-ENTROPY — how surprised the model was")
print("by the correct next token. Lower perplexity = lower cross-entropy.")

In [ ]:
# — Cell 5: Gradient Descent from Scratch ——————————————
# Gradient descent is HOW a model reduces its loss.
# We'll implement it with pure Python — no autograd, no PyTorch.

import numpy as np

np.random.seed(42)

# ── Dataset: y = 2x + 1 + noise ──────────────────────────
X = np.linspace(0, 10, 100)
y_true = 2.0 * X + 1.0 + np.random.normal(0, 0.5, size=X.shape)

# ── Model: y_hat = w*x + b ───────────────────────────────
# We have 2 parameters to learn: weight w and bias b
w = 0.0   # start with a bad guess
b = 0.0

LEARNING_RATE = 0.005
EPOCHS        = 200
N             = len(X)

history = []

for epoch in range(EPOCHS):
    # 1. PREDICT
    y_hat = w * X + b

    # 2. MEASURE (MSE loss)
    loss = np.mean((y_true - y_hat) ** 2)

    # 3. GRADIENTS — partial derivatives of loss w.r.t. w and b
    #    dL/dw = -2/N * sum(X * (y_true - y_hat))
    #    dL/db = -2/N * sum(y_true - y_hat)
    dw = (-2 / N) * np.sum(X * (y_true - y_hat))
    db = (-2 / N) * np.sum(y_true - y_hat)

    # 4. UPDATE weights in the direction that REDUCES loss
    w -= LEARNING_RATE * dw
    b -= LEARNING_RATE * db

    history.append(loss)

    if epoch % 40 == 0 or epoch == EPOCHS - 1:
        print(f"  Epoch {epoch:>3d}  |  loss={loss:.4f}  |  w={w:.4f}  b={b:.4f}")

print()
print(f"✓  Training complete!")
print(f"   Learned  : w = {w:.4f}, b = {b:.4f}")
print(f"   True     : w = 2.0000, b = 1.0000")
print()
print("This is EXACTLY what happens inside a neural network,")
print("except there are millions of w's and PyTorch computes")
print("the gradients automatically via autograd.")

In [ ]:
# — Cell 6: Visualise the Training Curve ———————————————
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: loss over epochs
axes[0].plot(history, color='#e74c3c', linewidth=2)
axes[0].set_title('Training Loss (MSE) over Epochs', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(alpha=0.3)
axes[0].annotate(f'Final: {history[-1]:.4f}',
                 xy=(len(history)-1, history[-1]),
                 xytext=(len(history)*0.6, history[0]*0.6),
                 arrowprops=dict(arrowstyle='->', color='black'),
                 fontsize=10)

# Right: learned line vs data
axes[1].scatter(X, y_true, alpha=0.4, s=15, color='#3498db', label='Data')
axes[1].plot(X, w*X+b, color='#e74c3c', linewidth=2.5, label=f'Learned: y={w:.2f}x+{b:.2f}')
axes[1].plot(X, 2*X+1, color='#2ecc71', linewidth=1.5, linestyle='--', label='True: y=2x+1')
axes[1].set_title('Data vs Learned Line', fontsize=12, fontweight='bold')
axes[1].set_xlabel('X')
axes[1].set_ylabel('y')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/ch1_training_curve.png', dpi=120, bbox_inches='tight')
plt.show()
print("✓  Plot saved to /content/ch1_training_curve.png")

In [ ]:
# — Cell 7: Learning Rate — The Most Important Hyperparameter
# The learning rate controls HOW BIG a step we take each update.
# Too large → overshoot and diverge. Too small → takes forever.

import numpy as np

def train(lr, epochs=150):
    np.random.seed(42)
    X = np.linspace(0, 10, 100)
    y = 2.0 * X + 1.0 + np.random.normal(0, 0.5, size=X.shape)
    w, b, N = 0.0, 0.0, len(X)
    losses = []
    for _ in range(epochs):
        y_hat = w*X + b
        loss  = np.mean((y - y_hat)**2)
        if np.isnan(loss) or loss > 1e6:
            losses.append(float('nan'))
            break
        losses.append(loss)
        dw = (-2/N)*np.sum(X*(y - y_hat))
        db = (-2/N)*np.sum(y - y_hat)
        w -= lr * dw
        b -= lr * db
    return losses

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))

configs = [
    (0.0001, '#3498db', 'Too small (0.0001) — barely moves'),
    (0.005,  '#2ecc71', 'Just right (0.005) — converges cleanly'),
    (0.05,   '#e74c3c', 'Too large (0.05)  — diverges / NaN'),
]

for lr, color, label in configs:
    losses = train(lr)
    valid  = [l for l in losses if not (isinstance(l, float) and l != l)]
    ax.plot(valid, color=color, linewidth=2, label=label)

ax.set_title('Learning Rate Comparison', fontsize=12, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (MSE)')
ax.set_ylim(0, 50)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/ch1_learning_rate.png', dpi=120, bbox_inches='tight')
plt.show()
print("✓  Plot saved to /content/ch1_learning_rate.png")
print()
print("Rule of thumb: start at 1e-3 and halve it if loss oscillates.")
print("For LLM fine-tuning (Ch6), we'll use 2e-4 — the standard QLoRA rate.")

In [ ]:
# — Cell 8: Overfitting, Underfitting & the Train/Val Split
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

# True relationship: y = sin(x)
X = np.sort(np.random.uniform(0, 2*np.pi, 30))
y = np.sin(X) + np.random.normal(0, 0.2, size=X.shape)

# Split: 70% train, 30% validation
split = int(0.7 * len(X))
X_train, y_train = X[:split], y[:split]
X_val,   y_val   = X[split:], y[split:]

X_plot = np.linspace(0, 2*np.pi, 200)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
titles = ['Underfit (degree 1)', 'Good fit (degree 4)', 'Overfit (degree 15)']
degrees = [1, 4, 15]

for ax, deg, title in zip(axes, degrees, titles):
    coeffs   = np.polyfit(X_train, y_train, deg)
    y_fit    = np.polyval(coeffs, X_plot)
    train_e  = np.mean((y_train - np.polyval(coeffs, X_train))**2)
    val_e    = np.mean((y_val   - np.polyval(coeffs, X_val  ))**2)

    ax.scatter(X_train, y_train, color='#3498db', s=30, label='Train data', zorder=3)
    ax.scatter(X_val,   y_val,   color='#e67e22', s=30, marker='D', label='Val data', zorder=3)
    ax.plot(X_plot, y_fit, color='#e74c3c', linewidth=2)
    ax.plot(X_plot, np.sin(X_plot), color='#2ecc71', linestyle='--', linewidth=1.5, label='True')
    ax.set_title(f'{title}\nTrain MSE={train_e:.3f}  Val MSE={val_e:.3f}', fontsize=10)
    ax.set_ylim(-2.5, 2.5)
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/ch1_overfit.png', dpi=120, bbox_inches='tight')
plt.show()
print("✓  Plot saved to /content/ch1_overfit.png")
print()
print("KEY TAKEAWAYS:")
print("  Underfit : model too simple — high train AND val error")
print("  Good fit : generalises well — low train AND val error")
print("  Overfit  : memorised training data — low train, HIGH val error")
print()
print("For LLMs: overfitting during fine-tuning is called CATASTROPHIC")
print("FORGETTING — the model loses general capability.")
print("We prevent it with small LoRA rank + few epochs (Ch6).")

In [ ]:
# — Cell 9: Chapter 1 — Vocabulary & Concept Map ————————

glossary = {
    "Parameter / Weight" : "A number inside the model that gets adjusted during training.",
    "Loss Function"      : "Measures how wrong the model's prediction is. Training minimises this.",
    "Gradient"           : "The direction and magnitude of the steepest increase in loss.",
    "Gradient Descent"   : "Move weights OPPOSITE to the gradient to reduce loss.",
    "Learning Rate (lr)" : "How big each weight update step is. Critical hyperparameter.",
    "Epoch"              : "One full pass through the entire training dataset.",
    "Batch"              : "A subset of training data processed together before one weight update.",
    "Overfitting"        : "Model memorises training data; fails on unseen data.",
    "Underfitting"       : "Model too simple to capture the pattern in the data.",
    "Train/Val Split"    : "Hold out some data to measure generalisation during training.",
    "Hyperparameter"     : "Settings YOU choose before training (lr, epochs, batch size).",
    "Inference"          : "Using a trained model to make predictions on new input.",
}

print("CHAPTER 1 — CORE VOCABULARY")
print("=" * 60)
for term, definition in glossary.items():
    print(f"  {term:<25}  {definition}")
print()
print("These terms appear in every paper, blog, and codebase you will read.")
print("Ch2 adds the data-science toolkit. Ch3 builds a neural network")
print("using these exact concepts. Ch6 applies them to fine-tuning TinyLlama.")

## ✓ Chapter 1 Complete

You now understand the foundation every ML practitioner builds on:
the training loop, loss functions, gradient descent, and overfitting.

**Next:** Chapter 2 — Python & Data Science Toolkit
You'll learn the data manipulation and visualisation tools used throughout
the rest of the course: numpy arrays, pandas DataFrames, and matplotlib.

---
*JAWNVION LLC AI Training Workbook · peter@jawnvion.com*